Нам нужны пакеты:
- `numpy`
- `pandas`
- `matplotlib`, `seaborn`
- `statsmodels`
- `linearmodels`
- `pmdarima`
- `scikit-learn`
- `scipy`
- `xlrd`, `openpyxl`

In [ ]:
import pandas as pd
import seaborn as se
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Парные регрессии

Для чтения файлов Excel нам дополнительно нужны пакеты
- `xlrd` для старых форматов,
- `openpyxl` для новых (которые `.xlsx`).

In [ ]:
data = pd.read_excel("../data1/cps12.xlsx")

Удостоверимся, что мы действительно что-то загрузили.

In [ ]:
data.head()

Воспользуемся «методом вдумчивого взгляда» и посмотрим на диаграмму рассеяния.

Будем использовать пакет `seaborn`.
Параметр `data=` показывает, из какого набора данных брать `x` и `y`.

In [ ]:
se.scatterplot(data=data, x="age", y="ahe")

In [ ]:
se.scatterplot(data=data, x="female", y="ahe")

Построим корреляционную матрицу `age` и `ahe`.
Я предпочитаю оборачивать все в функцию `print` чтобы избежать переформатирования со стороны Jupyter.

In [ ]:
print(data[["ahe", "age"]].corr())

Оценим модель.
Я покажу два с половиной способа это сделать.
«Под капотом» они делают одно и то же, но снаружи второй способ **лично мне** удобнее.

Первый способ состоит в прямом использовании класса `OLS`.
Нам надо вручную создавать матрицу объясняющих переменных, добавлять в нее константу.

Также, по-умолчанию, конструктор класса не проверяет данные на пропуски; если это не то, что нужно, то передайте параметру `missing=` значение `drop` для удаления наблюдений с пропусками или `raise` для вызова исключения.

Ниже мы последовательно:
1. Создаем матрицу экзогенных переменных (то есть объясняющих) и добавляем в нее свободный член (столбец из единиц).
2. Создаем объект `OLS`, передавая в него сначала объясняемую переменную, а затем матрицу экзогенных переменных.
3. Подгоняем модель (оцениваем ее).
4. Выводим общий результат.

In [ ]:
X = data["age"]
X = sm.add_constant(X)

model = sm.OLS(data.ahe, X)
model_fitted = model.fit()
print(model_fitted.summary())

Альтернативный способ состоит в применении метода `ols` из модуля `statsmodels.formula.api`.
Данный метод принимает на вход описание модели в стиле, принятом в `R`.
Оценивание проводится аналогично.

In [ ]:
model = smf.ols("ahe ~ age", data=data).fit()
print(model.summary())

Можно использовать функции и с классом `OLS`, но нужно не создавать объект напрямую, а вызвать метод `from_formula`.

In [ ]:
model = sm.OLS.from_formula("ahe ~ age", data=data).fit()
print(model.summary())

In [ ]:
model = sm.OLS.from_formula("ahe ~ female", data=data).fit()
print(model.summary())

In [ ]:
(
    data.loc[data.female == 0].ahe.mean(),
    data.loc[data.female == 1].ahe.mean(),
    data.loc[data.female == 1].ahe.mean() - data.loc[data.female == 0].ahe.mean(),
)